**Tess Vu**

This notebook is a quick derivation from Jill's original `newpred.ipynb` that establishes some fixes, much of it is not formatted and commented yet.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from shapely.geometry import box
import os

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

IN_SAL_2011 = RAW / "census_2011" / "ea_sal_kzn_gp.shp"
IN_WARDS = RAW / "wards_2020" / "SA_Wards2020.shp"
IN_WARD_POP = RAW / "wards_pop.csv"
IN_SAL_DEDUP = INTERMEDIATE / "sal_w_ward_dedup" / "sal_w_ward_dedup.shp"
IN_SAL_WARD_JOIN = RAW / "sal_w_ward_new" / "sal_w_ward_new.shp"
OUT_POP_PRED = INTERMEDIATE / "pop_pred_final.csv"


In [ ]:
print(os.getcwd())

In [ ]:
sal = gpd.read_file(IN_SAL_2011)

wards = gpd.read_file(IN_WARDS)

wards_with_pop = pd.read_csv(IN_WARD_POP, thousands = ",")

In [ ]:
pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_columns', 100)

In [ ]:
wards["WardID"]

In [ ]:
wards_with_pop["Ward_Code"]

In [ ]:
# Drop last row with "Total" in the Ward_Code.
wards_with_pop = wards_with_pop.drop(wards_with_pop.index[-1]) 

In [ ]:
wards_with_pop["Ward_Code"]

In [ ]:
wards_with_pop.rename(columns = {"Ward_Code": "WardID"}, inplace = True)
wards_with_pop.rename(columns = {" Total": "Total"}, inplace = True)

In [ ]:
wards = wards.merge(
    wards_with_pop[['WardID', 'Total']],
    on='WardID',
    how='left'
)

In [ ]:
wards = wards[wards['Province'].isin(['Gauteng', 'KwaZulu-Natal'])].copy()

In [ ]:
sal['population'].isna().sum()

In [ ]:
sal_with_ward = gpd.read_file(IN_SAL_DEDUP)

In [ ]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
sal_with_ward=sal_with_ward.rename(columns={'census_war': 'WardID'})

In [ ]:
sal_with_ward

In [ ]:
sal_with_ward = sal_with_ward.merge(
    wards[['WardID', 'Total']],
    on='WardID',
    how='left'
)

In [ ]:
sal_with_ward

In [ ]:
# SELECT COLUMNS AND DEDUPLICATE BY WARD OVERLAP
# Sort by AREA descending so the SAL-ward pair with the
# largest area overlap is kept during deduplication.
sal_wards = sal_with_ward[
    ["WardID", "EA_CODE", "sal2011_po", "Total", "EA_GTYPE", "EA_TYPE",
     "F4_class", "num_houses", "Black_Afri", "White", "Coloured",
     "Indian_or", "Other", "AREA", "Shape_Area"]
].copy()

# Assign each SAL to the ward with the largest geometric overlap.
sal_wards = (
    sal_wards.sort_values("AREA", ascending = False)
    .drop_duplicates(subset = "EA_CODE", keep = "first")
    .reset_index(drop = True)
)

print(f"SAL rows after dedup: {len(sal_wards)}")


In [ ]:
sal_wards = sal_wards.rename(columns = {
    "sal2011_po": "sal2011_pop",
    "Total": "ward2023_pop",
    "F4_class": "econ_status",
    "num_houses": "houses2011"
})

# Compute area_km2 from the SAL polygon area (Shape_Area is in CRS units squared).
# sal_w_ward_new.shp is in EPSG:32735 (UTM zone 35S), so Shape_Area is in m².
sal_wards["area_km2"] = sal_wards["Shape_Area"] / 1e6

# Drop helper columns no longer needed.
sal_wards = sal_wards.drop(columns = ["AREA", "Shape_Area"])

In [ ]:
sal_wards['EA_TYPE'] = sal_wards['EA_TYPE'].str.replace(r'_\*$', '', regex=True)
sal_wards['EA_TYPE'] = sal_wards['EA_TYPE'].str.replace(
    'Smallholdings', 'Small holdings'
)

In [ ]:
# ZERO-POPULATION IMPUTATION
# For SALs with 2011 pop of 0 but nonzero dwelling count, impute
# using Stats SA 2022 Census average household size of 3.3.
# SALs with both pop = 0 and houses = 0 remain at zero (truly vacant).
HH_SIZE = 3.3

zero_pop_mask = sal_wards["sal2011_pop"] == 0
has_houses_mask = sal_wards["houses2011"] > 0
impute_mask = zero_pop_mask & has_houses_mask

sal_wards.loc[impute_mask, "sal2011_pop"] = (
    sal_wards.loc[impute_mask, "houses2011"] * HH_SIZE
)

imputed_count = impute_mask.sum()
remaining_zero = (sal_wards["sal2011_pop"] == 0).sum()
print(f"Imputed {imputed_count} SALs with pop = 0 but houses > 0.")
print(f"Remaining zero-pop SALs (truly vacant): {remaining_zero}")

# DENSITY COLUMNS (for EDA, not used in step-down weight).
sal_wards["sal_dense"] = (
    sal_wards["sal2011_pop"].astype(float) /
    sal_wards["area_km2"].astype(float)
)
sal_wards["log_density"] = np.log1p(sal_wards["sal_dense"])


In [ ]:
sal_wards

In [ ]:
sal_wards['log_density'].describe()

In [ ]:
# Coerce to numeric in case of string remnants from CSV reads.
sal_wards["ward2023_pop"] = pd.to_numeric(sal_wards["ward2023_pop"], errors = "coerce")
sal_wards["sal2011_pop"] = pd.to_numeric(sal_wards["sal2011_pop"], errors = "coerce")

In [ ]:
# VALIDATION: Confirm no duplicate EA_CODEs remain.
dupes = sal_wards[sal_wards.duplicated(subset = "EA_CODE", keep = False)]
print(f"Duplicate EA_CODEs remaining: {len(dupes)}")

assert len(dupes) == 0, "Unexpected duplicates after dedup step."

In [ ]:
# PROPORTIONAL STEP-DOWN WEIGHT
# Compute each SAL's share of its parent ward's 2011 population.
# This is the standard step-down method (Luo & Wang 2003).
ward2011_sum = sal_wards.groupby("WardID", as_index = False)["sal2011_pop"].sum()
ward2011_sum = ward2011_sum.rename(columns = {"sal2011_pop": "ward2011_sum"})

sal_wards = sal_wards.merge(
    ward2011_sum,
    on = "WardID",
    how = "left"
)

sal_wards["share2011"] = sal_wards["sal2011_pop"] / sal_wards["ward2011_sum"]

# Shares should sum to 1 within each ward (pycnophylactic constraint).
ward_share_sums = sal_wards.groupby("WardID")["share2011"].sum()
print(f"Ward share sums — min: {ward_share_sums.min():.6f}, max: {ward_share_sums.max():.6f}")


In [ ]:
sal_wards['share2011'].describe()

In [ ]:
# Note: log_density is retained for EDA but NOT used in the step-down weight.
# Using share2011 alone avoids double-counting the 2011 population signal
# (share2011 and log_density are both derived from sal2011_pop).


In [ ]:
# sal_wards['gweight'].describe()

In [ ]:
# STEP-DOWN WEIGHT = share2011 (proportional allocation).
# No density re-weighting, avoids circularity with 2011 pop.
sal_wards["dasym_weight"] = sal_wards["share2011"]

In [ ]:
sal_wards['dasym_weight'].describe()

In [ ]:
# ESTIMATE 2023 SAL POPULATION
# Distribute 2023 ward population to SALs using proportional weights.
sal_wards["sal2023_est"] = sal_wards["dasym_weight"] * sal_wards["ward2023_pop"]

total_est = sal_wards["sal2023_est"].sum()
print(f"Total estimated 2023 pop: {total_est:,.0f}")

In [ ]:
# COMPOUND ANNUAL GROWTH RATE (2011-2023, 12 years).
sal_wards["growth_rate"] = (
    (sal_wards["sal2023_est"] / sal_wards["sal2011_pop"]) ** (1 / 12)
) - 1

In [ ]:
sal_wards['ward2023_pop'].describe()

In [ ]:
sal_wards['sal2023_est'].describe()

In [ ]:
sal_wards['sal2023_est'].sum()

In [ ]:
wards["ward2023_pop"] = pd.to_numeric(wards["Total"], errors = "coerce")

In [ ]:
wards['ward2023_pop'].sum()

In [ ]:
sal_wards['sal2023_est'].sum()-wards['ward2023_pop'].sum() 

In [ ]:
sal_wards[['sal2011_pop','sal2023_est', ]].sum()

In [ ]:
sal_wards['EA_CODE'] = sal_wards['EA_CODE'].astype('Int64')   # or str if needed

In [ ]:
sal_wards

In [ ]:
print(sal_wards.head().to_markdown(index=False, floatfmt=",.2f"))

In [ ]:
gtype_summary = sal_wards.groupby('EA_TYPE').agg(
    pop2011=('sal2011_pop','sum'),
    pop2023=('sal2023_est','sum'),
).reset_index()

gtype_summary['growth_rate_2011_2023'] = (
    ((gtype_summary['pop2023'] / gtype_summary['pop2011']) ** (1/12) - 1)*100
)

In [ ]:
sal_wards['sal_dense'].describe()

In [ ]:
gtype_summary

In [ ]:
#print(gtype_summary.to_markdown(index=False, floatfmt=",.2f"))

In [ ]:
sal["EA_CODE"]

In [ ]:
sal_wards["EA_CODE"]

In [ ]:
# Normalize EA_CODE to plain string digits on both sides to avoid dtype mismatch.
# .astype(str).str.split('.').str[0] handles object, int, and float-like values
# (e.g. "123456.0" → "123456") without any integer casting that can fail.
sal['EA_CODE'] = sal['EA_CODE'].astype(str).str.strip().str.split('.').str[0]
sal_wards['EA_CODE'] = sal_wards['EA_CODE'].astype(str).str.strip().str.split('.').str[0]

# Drop columns from sal_wards that already exist in sal to avoid duplicate merge conflicts.
# Keep only the new/computed columns and EA_CODE for the join.
cols_to_keep = ["EA_CODE", "WardID", "sal2011_pop", "ward2023_pop", "econ_status", 
                "houses2011", "Black_Afri", "White", "Coloured", "Indian_or", "Other",
                "area_km2", "dasym_weight", "share2011", "sal_dense", "log_density", 
                "sal2023_est", "growth_rate", "ward2011_sum"]

sal_map = sal.merge(
    sal_wards[cols_to_keep],
    on='EA_CODE',
    how='left'
)


In [ ]:
sal_wards['sal2023_est'].describe()

In [ ]:
sal_map['dense_2023']= sal_map['sal2023_est']/sal_map['area_km2']

In [ ]:
sal_map['dense_2023'].describe()

In [ ]:
sal_map['log_density'] = np.log1p(sal_map['dense_2023'])

In [ ]:
sal_map['log_density'].describe()

In [ ]:
sal_wards.to_csv(OUT_POP_PRED, index=False)

In [ ]:
sal_map = sal_map.to_crs(sal_with_ward.crs)

In [ ]:
sal_wards['sal_dense'].describe()

In [ ]:
print(sal_with_ward.crs)

In [ ]:
print(sal_wards['share2011'].describe().to_markdown(index=False, floatfmt=",.2f"))

In [ ]:
sal_wards[sal_wards['sal_dense'] >100000].sort_values('area_km2')

In [ ]:
sal_wards.columns

In [ ]:
sal_with_ward.columns

In [ ]:
cols = ['sal2023_est', 'sal2011_pop', 'ward2023_pop', 'ward2011_sum', 'growth_rate', 
        'dasym_weight', 'share2011', 'log_density', 'sal_dense']

summary = sal_wards[cols].describe()

print(summary.to_markdown(floatfmt=",.3f"))

In [ ]:
sal_wards['growth_rate'].hist()

In [ ]:
print(sal_wards.loc[sal_wards['growth_rate'].idxmin()])

In [ ]:
sal_wards.loc[sal_wards['sal_dense'].idxmax()]

In [ ]:
sal_with_ward = pd.read_csv(OUT_POP_PRED)

In [ ]:
sal_with_ward= sal_with_ward.merge(
    sal, 
    on="EA_CODE", 
   )
sal_with_ward = gpd.GeoDataFrame(
    sal_with_ward,
   geometry="geometry")

## Cross-Checks Against the Deduplicated SAL Layer

These checks moved here from `sal_w_ward_deduplication.ipynb`, which no longer reads `pop_pred_final`.


In [ ]:
# Setup for the moved checks: reload the raw join and NB2's own output,
# then rebuild NB1's deduplication bookkeeping.
sal_raw = gpd.read_file(IN_SAL_WARD_JOIN)
pop = pd.read_csv(OUT_POP_PRED)

dup_counts = sal_raw.groupby("EA_CODE").size()
dup_ea_codes = dup_counts[dup_counts > 1].index
ward_col = "census_war"

sal_dedup = (
    sal_raw
    .sort_values("AREA", ascending = False)
    .drop_duplicates(subset = ["EA_CODE"], keep = "first")
    .sort_index()
    .copy()
)


In [ ]:
# PRE-DEDUPLICATION CROSS-CHECK 5: WARD MATCH AGAINST pop_pred_final.
# After deduplication, the retained WardID for each EA_CODE should match
# the WardID in pop_pred_final.csv, which was built from the same spatial
# join using the same dominant-ward logic. A mismatch here means the
# deduplication strategy diverges from how the population model assigned wards.

# Build a lookup from pop_pred_final: EA_CODE -> WardID.
pop_ward_lookup = pop.set_index("EA_CODE")["WardID"]

# Simulate deduplication to get the retained WardID per EA_CODE.
sal_sorted = sal_raw.sort_values("AREA", ascending = False)
retained = sal_sorted.drop_duplicates(subset = ["EA_CODE"], keep = "first")[["EA_CODE", ward_col]].copy()
retained = retained.rename(columns = {ward_col: "shp_WardID"})
retained["shp_WardID"] = retained["shp_WardID"].astype(str)

# Align types for comparison.
pop_ward_series = pop_ward_lookup.astype(str)
retained["pop_WardID"] = retained["EA_CODE"].map(pop_ward_series)

# Only check EA_CODEs that were actually duplicated (single-occurrence
# EA_CODEs are trivially matched).
retained_dups = retained[retained["EA_CODE"].isin(dup_ea_codes)].copy()
ward_matches   = (retained_dups["shp_WardID"] == retained_dups["pop_WardID"]).sum()
ward_mismatches = (retained_dups["shp_WardID"] != retained_dups["pop_WardID"]).sum()

print("WARD MATCH AGAINST pop_pred_final (duplicated EA_CODEs only)")
print(f"  Retained WardID matches pop_pred_final: {ward_matches} / {len(retained_dups)}")
print(f"  Mismatches: {ward_mismatches}")

if ward_mismatches > 0:
    mismatch_rows = retained_dups[retained_dups["shp_WardID"] != retained_dups["pop_WardID"]].head(10)
    print()
    print("  Sample mismatches (shp_WardID vs pop_WardID):")
    print(mismatch_rows.to_string(index = False))
    print()
    print("  RECOMMENDATION: If mismatches are material, consider using PERCENTAGE")
    print("  instead of AREA as the deduplication key, or adopt the WardID from")
    print("  pop_pred_final directly via a post-deduplication merge.")
print()

In [ ]:
# POST-DEDUPLICATION CROSS-CHECK: EA_CODE MEMBERSHIP.
pop_codes = set(pop["EA_CODE"].unique())
sal_codes = set(sal_dedup["EA_CODE"].unique())

only_in_sal = sal_codes - pop_codes
only_in_pop = pop_codes - sal_codes

print("POST-DEDUPLICATION EA_CODE MEMBERSHIP CHECK")
print(f"  EA_CODEs in shapefile but NOT in pop_pred_final: {len(only_in_sal)}")
print(f"  EA_CODEs in pop_pred_final but NOT in shapefile: {len(only_in_pop)}")

if len(only_in_sal) > 0:
    print(f"  Sample (shapefile-only): {list(only_in_sal)[:5]}")
if len(only_in_pop) > 0:
    print(f"  Sample (pop-only): {list(only_in_pop)[:5]}")
